<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 El Buzón del Pueblo y el Chisme que se Riega: Flujos y Grafos 📬
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Big Data
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 05
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Big%20Data/05%20-%20Avances%20y%20Tendencias%20en%20Big%20Data/Para%20Dummies/00_Stream_y_Grafos_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 00 del Módulo 05 (flujos y grafos)](../00_Stream_Processing_y_Graph_Processing.ipynb). Imagina un **pueblo de Boyacá** con dos situaciones muy cotidianas:

| En el pueblo | En el mundo de los datos |
|---|---|
| Los pedidos llegan **por correo**, con el **sello de la fecha en que se enviaron**, pero el cartero **no los entrega en orden**: algunos se demoran horas. La tienda quiere **contar los pedidos de cada hora** sin esperar para siempre | **Procesamiento de flujos**: datos que llegan **sin parar y desordenados**; se agrupan en **ventanas de tiempo** y hay que decidir **cuánto esperar** a los rezagados (*marca de agua*) |
| Quieres saber **quién es la persona más importante del pueblo**, no por cuánta gente conoce, sino por **a quién le piden consejo los que a su vez son importantes** | **Grafos y PageRank**: la importancia **se hereda** de quienes te recomiendan |
| Un **chisme** se riega: en cada tanda, quien lo sabe se lo cuenta a sus vecinos. ¿**En cuántas tandas** se entera todo el pueblo? | **Pregel**: los grafos enormes se procesan **por tandas**, y en cada tanda cada nodo **solo habla con sus vecinos** |

Al terminar podrás explicar:
1. Por qué **contar por la fecha del sello** (cuando ocurrió) es más justo que contar por la **hora de llegada**, y qué se **gana** y qué se **pierde** al esperar más.
2. Por qué **la importancia de una persona no es lo mismo que tener muchos conocidos**.
3. Por qué en un grafo enorme conviene que cada nodo **hable solo con sus vecinos** y se avance **por tandas**.

> 🔗 Si luego quieres ver el **log tipo Kafka**, Spark Structured Streaming real, PageRank verificado contra `networkx` y Pregel desde cero: [cuaderno estándar](../00_Stream_Processing_y_Graph_Processing.ipynb). Y al terminar este: [la báscula, los tres colegios y el cuaderno con sellos](01_Tendencias_IA_Blockchain_Dummies.ipynb).

---
## 1. El Buzón del Pueblo: Contar por Hora del Sello 📬

La tienda de don Álvaro recibe **12 pedidos por correo** entre las 8 y las 12 de la mañana (datos **inventados**). Cada carta trae el **sello con la hora en que se envió** y nosotros anotamos **a qué hora llegó**. Las horas se escriben en **decimales** (8,5 = 8:30 a. m.).

Queremos contar **cuántos pedidos se enviaron en cada hora** (de 8 a 9, de 9 a 10...). Dos formas:

1. **Contar por la hora de llegada** (la más fácil, pero **injusta**: una carta enviada a las 8:54 que llega a las 9:12 se cuenta en la hora equivocada).
2. **Contar por la hora del sello** y **esperar un rato** antes de cerrar cada hora para dar tiempo a las cartas demoradas (esto es lo que hace la **marca de agua**).

Mira cuántos pedidos **quedan bien contados** según **cuánto esperes**.

In [ ]:
import math

import matplotlib.pyplot as plt
import pandas as pd

# (hora del sello, hora de llegada) de 12 pedidos inventados
cartas = [(8.1, 8.3), (8.5, 8.8), (8.9, 9.2), (9.2, 9.4), (9.6, 9.7), (9.9, 10.5),
          (10.2, 10.4), (10.7, 10.9), (10.8, 13.3), (11.1, 11.3), (11.5, 11.6), (11.9, 12.4)]
assert len(cartas) == 12

# 1) Contar por hora de LLEGADA (injusto)
por_llegada = pd.Series([math.floor(llegada) for _, llegada in cartas]).value_counts().sort_index()
# 2) Contar por hora del SELLO (lo correcto, si esperamos lo suficiente)
por_sello = pd.Series([math.floor(sello) for sello, _ in cartas]).value_counts().sort_index()
print("Pedidos por hora de LLEGADA:", por_llegada.to_dict())
print("Pedidos por hora del SELLO :", por_sello.to_dict())
assert por_sello.to_dict() == {8: 3, 9: 3, 10: 3, 11: 3}
assert por_llegada.to_dict() != por_sello.to_dict()                              # contar por llegada da OTRA historia


def contar_con_espera(espera):
    """Cuenta por hora del sello, pero cierra cada hora 'espera' horas despues de su fin: lo que llega mas tarde se ignora."""
    conteo = {8: 0, 9: 0, 10: 0, 11: 0}
    for sello, llegada in cartas:
        hora = math.floor(sello)
        if llegada <= hora + 1 + espera:                                         # llego antes de que se cerrara su hora
            conteo[hora] += 1
    return conteo


filas = []
for espera in [0, 0.5, 1, 2, 3]:
    c = contar_con_espera(espera)
    filas.append({"espera (horas)": espera, "pedidos contados": sum(c.values()), "de 12": f"{sum(c.values()) / 12:.0%}"})
tabla = pd.DataFrame(filas)
print("\n" + tabla.to_string(index=False))
assert tabla["pedidos contados"].tolist() == [8, 11, 11, 11, 12]                 # esperar mas = contar mas, pero el resultado sale mas tarde
assert contar_con_espera(3) == por_sello.to_dict()

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(tabla["espera (horas)"].astype(str), tabla["pedidos contados"], color=["#dc2626", "#f59e0b", "#f59e0b", "#f59e0b", "#16a34a"])
ax.axhline(12, color="k", ls="--", lw=1); ax.text(-0.4, 12.2, "los 12 pedidos reales", fontsize=8)
ax.set_xlabel("cuánto se espera antes de cerrar cada hora (horas)"); ax.set_ylabel("pedidos bien contados")
ax.set_title("Esperar más cuenta más pedidos... pero el reporte sale más tarde", fontsize=10, fontweight="bold")
plt.tight_layout(); plt.show()

---
##### 🛠️ Práctica 1: ¿Cuánto hay que esperar para no perder ninguna carta?

Con la lista `cartas` y la función `contar_con_espera` del código anterior:

1. Guarda en `candidatos` la lista `[0, 0.5, 1, 2, 3]`.
2. Guarda en `espera_min` la **menor espera** de `candidatos` con la que se cuentan **los 12 pedidos** (pista: `sum(contar_con_espera(e).values()) == 12`).
3. Comprueba con `assert` que `espera_min == 3` y que con la espera **anterior** de la lista **no** se cuentan los 12.

In [ ]:
# Escribe tu código aquí

# candidatos = [...]
# espera_min = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
candidatos = [0, 0.5, 1, 2, 3]
espera_min = next(e for e in candidatos if sum(contar_con_espera(e).values()) == 12)
print("Hay que esperar al menos", espera_min, "horas (la carta más demorada tardó 2,5 horas en llegar).")

assert espera_min == 3
assert sum(contar_con_espera(2).values()) < 12
```
</details>

---


---
**Lección:** si **no esperas nada**, pierdes **4 de 12** pedidos (los de las cartas demoradas); con **media hora** ya rescatas **11**, y para no perder **ninguno** hay que esperar **hasta que llegue la carta más tardona** (aquí 2,5 horas, o sea, «3» en nuestra lista). Esa es la **elección de la marca de agua**: **más espera = más exactitud, pero el reporte sale más tarde**. No hay respuesta perfecta; se elige según lo que cueste **equivocarse** frente a lo que cueste **esperar**.

---

## 2. ¿Quién es la Persona más Importante del Pueblo? 🏘️

En el pueblo hay **6 personas** y cada una dice **a quién le pide consejo**:

| Persona | Le pide consejo a... |
|---|---|
| **Rosa** (la tendera) | Padre Luis |
| **Padre Luis** | Rosa |
| **Beto** | Rosa y Padre Luis |
| **Carla** | Rosa |
| **Diego** | Rosa y Beto |
| **Elena** | Carla |

**Idea:** cada persona empieza con **100 puntos de «importancia»**. En **cada ronda** reparte **el 85 %** de sus puntos **por partes iguales** entre las personas a quienes pide consejo, y el **15 %** restante se reparte **entre todo el pueblo** (porque a veces uno le pregunta a cualquiera). Después de varias rondas los puntos **dejan de cambiar**: eso es la **importancia**. Recibir consejos de alguien **muy importante** vale **más** que de muchas personas poco importantes. (Esta idea es **PageRank**, el corazón del buscador original de Google.)

In [ ]:
personas = ["Rosa", "Padre Luis", "Beto", "Carla", "Diego", "Elena"]
pide_consejo_a = {"Rosa": ["Padre Luis"], "Padre Luis": ["Rosa"], "Beto": ["Rosa", "Padre Luis"],
                  "Carla": ["Rosa"], "Diego": ["Rosa", "Beto"], "Elena": ["Carla"]}


def una_ronda(puntos):
    nuevos = {p: 0.15 * sum(puntos.values()) / len(personas) for p in personas}          # el 15 %: se reparte entre todos
    for p, destinos in pide_consejo_a.items():
        for d in destinos:
            nuevos[d] += 0.85 * puntos[p] / len(destinos)                                  # el 85 %: se reparte entre sus consejeros
    return nuevos


puntos = {p: 100.0 for p in personas}
historia = [puntos]
for ronda in range(150):
    puntos = una_ronda(puntos)
    historia.append(puntos)

print("Puntos después de 150 rondas:")
for p, v in sorted(puntos.items(), key=lambda kv: -kv[1]):
    print(f"  {p:<11} {v:6.1f}")
assert abs(sum(puntos.values()) - 600) < 1e-9                                            # los puntos se reparten, no se crean ni se pierden
assert max(abs(historia[-1][p] - historia[-2][p]) for p in personas) < 1e-6              # ya no cambian: llegamos a la importancia
assert max(puntos, key=puntos.get) == "Rosa"                                             # todos le piden consejo a Rosa o a quien se lo pide a Rosa

# Importancia no es lo mismo que conocidos: cuantas personas le piden consejo directamente a cada uno
directos = {p: sum(p in d for d in pide_consejo_a.values()) for p in personas}
print("\nPersonas que le piden consejo directamente:", directos)
assert directos["Rosa"] == 4 and directos["Padre Luis"] == 2 and directos["Beto"] == 1 and directos["Carla"] == 1
assert puntos["Padre Luis"] > 10 * puntos["Beto"]                                         # el Padre tiene solo 2 admiradores, pero uno es Rosa: vale mucho mas que Beto

# Verificacion de un caso con respuesta conocida: un circulo perfecto de 3 amigos que se piden consejo en cadena
circulo = {"A": ["B"], "B": ["C"], "C": ["A"]}
pts = {"A": 100.0, "B": 100.0, "C": 100.0}
for _ in range(20):
    nuevo = {p: 0.15 * sum(pts.values()) / 3 for p in pts}
    for p, ds in circulo.items():
        nuevo[ds[0]] += 0.85 * pts[p]
    pts = nuevo
assert all(abs(v - 100) < 1e-9 for v in pts.values())                                    # el circulo es simetrico: nadie manda mas

fig, ax = plt.subplots(figsize=(7.5, 3.8))
for p in personas:
    ax.plot(range(len(historia)), [h[p] for h in historia], label=p)
ax.set_xlabel("ronda"); ax.set_ylabel("puntos de importancia"); ax.legend(fontsize=8, ncol=2)
ax.set_title("Los puntos se acomodan y dejan de cambiar: eso es la importancia", fontsize=10, fontweight="bold")
plt.tight_layout(); plt.show()

---
**Lección:** Rosa queda **primera** (cuatro de las cinco personas restantes le piden consejo). Pero fíjate en el **Padre Luis**: solo **2 personas** le piden consejo, **menos que Rosa (4)**, y aun así **vale más de diez veces lo que vale Beto**, porque **uno de sus dos admiradores es Rosa**. **La importancia se hereda**: no cuenta **cuántos** te siguen, sino **quiénes**. (Y en el círculo de tres amigos que se piden consejo en cadena, **nadie** manda más: todos quedan en 100.)

---

## 3. El Chisme que se Riega por Tandas 🗣️

Un chisme nace en **Elena**. En **cada tanda**, **todos los que ya lo saben** se lo cuentan **a los vecinos** (las personas con las que hablan: aquí, **las mismas conexiones de arriba**, pero sin importar quién le pide consejo a quién). Un grafo gigante se procesa así: **por tandas**, y en cada tanda **cada nodo solo habla con sus vecinos**, sin que nadie vea el pueblo completo. Contemos **cuántas tandas** hacen falta y **cuántas personas** lo saben en cada una.

In [ ]:
# Vecinos: con quien habla cada persona (en ambos sentidos)
vecinos = {p: set() for p in personas}
for p, destinos in pide_consejo_a.items():
    for d in destinos:
        vecinos[p].add(d); vecinos[d].add(p)

saben = {"Elena"}
tandas = [set(saben)]
while True:
    nuevos = {v for p in saben for v in vecinos[p]} - saben                              # en esta tanda se enteran los vecinos de quienes ya saben
    if not nuevos:
        break
    saben |= nuevos
    tandas.append(set(saben))

for t, s in enumerate(tandas):
    print(f"Tanda {t}: lo saben {len(s)} -> {sorted(s)}")
assert len(tandas) - 1 == 3                                                              # 3 tandas hasta el ultimo (Padre Luis y Beto, Diego)
assert tandas[-1] == set(personas) and [len(s) for s in tandas] == [1, 2, 3, 6]          # Elena; + Carla; + Rosa; + todos los demas

fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.bar(range(len(tandas)), [len(s) for s in tandas], color="#f59e0b")
ax.set_xticks(range(len(tandas))); ax.set_xlabel("tanda"); ax.set_ylabel("personas que ya lo saben"); ax.set_ylim(0, 6.5)
ax.set_title("El chisme llega a todo el pueblo en 3 tandas", fontsize=10, fontweight="bold")
plt.tight_layout(); plt.show()

---
**Lección:** el chisme llega a **todo el pueblo en 3 tandas**: Elena → Carla → Rosa → todos los demás. **Nadie necesitó conocer el pueblo completo**; cada quien solo habló con sus vecinos, **por tandas**. Esa es la forma en que se procesan los **grafos enormes** (redes sociales, páginas web): como **no cabe** el mapa completo en una sola máquina, cada nodo **recibe mensajes de sus vecinos, actualiza lo que sabe y manda mensajes**, **tanda tras tanda**, hasta que **ya no hay nada nuevo que contar**.

---

## Resumen en una frase 🎯

> **Un flujo de datos llega desordenado, así que se cuenta por el sello de cuándo ocurrió y se decide cuánto esperar a los rezagados; y en un grafo la importancia se hereda de quienes te recomiendan, y los grafos enormes se procesan por tandas donde cada nodo solo habla con sus vecinos.**

### Lo que aprendiste hoy:
- ✅ **El buzón del pueblo:** sin esperar, se perdían **4 de 12** pedidos; con media hora, 11; para no perder ninguno había que esperar **3 horas** (más espera = más exactitud, pero más demora).
- ✅ **La persona importante:** Rosa quedó primera; el **Padre Luis** vale **más de diez veces** lo que vale Beto con solo **2 admiradores**, porque uno es Rosa: **la importancia se hereda**.
- ✅ **El chisme por tandas:** llegó a los **6 vecinos en 3 tandas**, hablando cada quien solo con sus vecinos.

> 🎓 **Siguiente paso:** [el cuaderno estándar](../00_Stream_Processing_y_Graph_Processing.ipynb) implementa las ventanas, la marca de agua, un log tipo Kafka, PageRank y Pregel, y los verifica. Luego, [la báscula que se descalibra, tres colegios y un cuaderno con sellos](01_Tendencias_IA_Blockchain_Dummies.ipynb).

---
### 🧠 ¿Ya lo tienes claro?

1. Si la tienda de don Álvaro **necesita el reporte a las 12:05 en punto** y no puede esperar, ¿qué parte de los pedidos aceptará perder? ¿Y si el reporte fuera para **cobrar** los pedidos? ¿Esperarías más o menos?
2. En el pueblo, ¿qué pasaría con la importancia de Rosa si **deja de aconsejarse con el Padre Luis** y no se aconseja con nadie? ¿Y si Beto empezara a pedirle consejo a Elena?
3. Piensa en una **red de WhatsApp de tu vereda o barrio**: ¿quién sería «Rosa»? ¿En cuántas tandas se regaría una noticia desde tu casa?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Big Data (Edición Didáctica Para Dummies)</i>
  </p>
</div>